# Lecture 32 Worksheet: Self-Attention and Position
**NAVTTC / NIAI AI/ML Trainer Cohort** — runs in Google Colab (PyTorch is already installed) or a local Jupyter. Fill every `None` / `___` / `# YOUR CODE` area, then run top to bottom. Check cells print `OK` or tell you what to fix.

In Lecture 31 attention sat on top of an RNN. Today we meet **self-attention**: words in ONE sentence look at each other, with no RNN at all. You will build positional encoding and scaled dot-product attention yourself, test them against PyTorch's own, and then **measure** why position information is needed. Everything runs on a CPU in a few minutes.

In [ ]:
import os, math, json, time, random
os.environ["OMP_NUM_THREADS"] = "2"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
torch.set_num_threads(2); np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(0); np.random.seed(0); random.seed(0)
R = {}          # results we collect, printed as one JSON line at the very end
def check(name, ok, hint=""):
    print(("OK   " if ok else "FIX  ") + name + ("" if ok else "  -> " + hint))
print("torch", torch.__version__)

torch 2.11.0+cpu


## Our running sentence
`the child drinks milk` (4 words). Tokens → IDs → vectors. We use a tiny vocabulary of our own.

## Task 1: Tokens → IDs → embeddings

Build a vocabulary dict (word → ID, starting at 0, in the order given), turn the sentence into a LongTensor of shape (1, 4), and pass it through `nn.Embedding(len(vocab), 8)`.

In [ ]:
words = ["the", "child", "drinks", "milk", "boy", "reads", "book"]
vocab = {w: i for i, w in enumerate(words)}
ids = torch.tensor([[vocab[w] for w in ["the", "child", "drinks", "milk"]]], dtype=torch.long)
emb = nn.Embedding(len(words), 8)
x = emb(ids)

In [ ]:
check("vocab ids", vocab == {w: i for i, w in enumerate(words)})
check("ids shape", tuple(ids.shape) == (1, 4) and ids.dtype == torch.long, "LongTensor, shape (1, 4)")
check("embedding shape", tuple(x.shape) == (1, 4, 8))
print("shape (batch, seq_len, d_model) =", tuple(x.shape))

OK   vocab ids
OK   ids shape
OK   embedding shape
shape (batch, seq_len, d_model) = (1, 4, 8)


## Task 2: Sinusoidal positional encoding

Implement the formula from the notes (Page 4):

`PE(pos, 2i) = sin(pos / 10000^(2i/d))`  and  `PE(pos, 2i+1) = cos(pos / 10000^(2i/d))`

Return a tensor of shape `(max_len, d_model)`. Even columns get sin, odd columns get cos.

In [ ]:
def sinusoidal_pe(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

In [ ]:
pe = sinusoidal_pe(50, 6)
check("shape", tuple(pe.shape) == (50, 6))
check("position 0 is [0,1,0,1,0,1]", torch.allclose(pe[0], torch.tensor([0., 1, 0, 1, 0, 1])))
check("pos 1, dims 0/1 = sin(1), cos(1)", abs(pe[1, 0] - math.sin(1)) < 1e-6 and abs(pe[1, 1] - math.cos(1)) < 1e-6)
check("pos 2, dim 2 = sin(2 / 10000^(2/6))", abs(pe[2, 2] - math.sin(2 / 10000 ** (2 / 6))) < 1e-6)
check("every position has a different vector", len({tuple(r.tolist()) for r in pe.round(decimals=4)}) == 50)
print(pe[:4])

OK   shape
OK   position 0 is [0,1,0,1,0,1]
OK   pos 1, dims 0/1 = sin(1), cos(1)
OK   pos 2, dim 2 = sin(2 / 10000^(2/6))
OK   every position has a different vector
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000]])


### A picture of positional encoding
Each row is a position, each column a dimension. Low dimensions flip fast (high frequency), high dimensions flip slowly. That is how every position gets its own pattern.

In [ ]:
big = sinusoidal_pe(40, 32)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
im = ax.imshow(big.numpy(), cmap="RdBu_r", aspect="auto", vmin=-1, vmax=1)
ax.set_xlabel("dimension (0 to 31)"); ax.set_ylabel("position (0 to 39)")
ax.set_title("Sinusoidal positional encoding, d_model = 32")
fig.colorbar(im, ax=ax, fraction=0.04); fig.tight_layout(); fig.savefig("fig_pe_heat.png", dpi=170); plt.close(fig)
print("saved fig_pe_heat.png")

saved fig_pe_heat.png


## Task 3: Scaled dot-product attention

One formula, from Page 6 (and the same maths as Lecture 31's scaled-dot score):

`Attention(Q, K, V) = softmax(Q Kᵀ / sqrt(d_k)) V`

`Q`, `K`, `V` have shape `(batch, n, d_k)`. Return both the output and the weights. If `mask` is given (True = keep), put `-inf` on the masked scores BEFORE the softmax.

In [ ]:
def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    out = torch.matmul(weights, V)
    return out, weights

In [ ]:
# Page 6's tiny example: Q=[1,0], K=[1,0], V=[2,3], d_k = 2  ->  scaled score 1/sqrt(2) = 0.707, softmax over ONE key = 1, output = V
o, w = attention(torch.tensor([[[1., 0.]]]), torch.tensor([[[1., 0.]]]), torch.tensor([[[2., 3.]]]))
check("Page 6 example output [2, 3]", torch.allclose(o, torch.tensor([[[2., 3.]]])) and abs(w.item() - 1) < 1e-6)
g = torch.Generator().manual_seed(1)
Q, K, V = (torch.randn(2, 5, 16, generator=g) for _ in range(3))
o, w = attention(Q, K, V)
check("shapes", tuple(o.shape) == (2, 5, 16) and tuple(w.shape) == (2, 5, 5))
check("each row of weights sums to 1", torch.allclose(w.sum(-1), torch.ones(2, 5), atol=1e-5))
ref = F.scaled_dot_product_attention(Q, K, V)
check("matches PyTorch's own F.scaled_dot_product_attention", torch.allclose(o, ref, atol=1e-5), "check the sqrt(d_k) and the softmax dimension")
m = torch.tril(torch.ones(5, 5, dtype=torch.bool))
o2, w2 = attention(Q, K, V, mask=m)
check("masked weights above the diagonal are 0", float(w2.triu(1).abs().max()) == 0.0)
check("masked version matches PyTorch causal", torch.allclose(o2, F.scaled_dot_product_attention(Q, K, V, is_causal=True), atol=1e-5))

OK   Page 6 example output [2, 3]
OK   shapes
OK   each row of weights sums to 1
OK   matches PyTorch's own F.scaled_dot_product_attention
OK   masked weights above the diagonal are 0
OK   masked version matches PyTorch causal


## Task 4: A self-attention layer

Self-attention means Q, K and V all come from the **same** input `x`, through three learnable matrices `W_q`, `W_k`, `W_v` (Page 5). Build the module with `nn.Linear(d, d, bias=False)` for each.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W_q = nn.Linear(d, d, bias=False)
        self.W_k = nn.Linear(d, d, bias=False)
        self.W_v = nn.Linear(d, d, bias=False)
    def forward(self, x, mask=None):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        out, weights = attention(Q, K, V, mask=mask)
        return out, weights

In [ ]:
sa = SelfAttention(8)
out, w = sa(x)
check("output keeps the shape (1, 4, 8)", tuple(out.shape) == (1, 4, 8))
check("weights are (1, 4, 4): every word looks at every word", tuple(w.shape) == (1, 4, 4))
check("3 matrices of 8x8 = 192 parameters", sum(p.numel() for p in sa.parameters()) == 192)
manual = (F.softmax((x @ sa.W_q.weight.T) @ (x @ sa.W_k.weight.T).transpose(-2, -1) / math.sqrt(8), -1)) @ (x @ sa.W_v.weight.T)
check("equals the formula written out by hand", torch.allclose(out, manual, atol=1e-5))
print("attention weights of an UNTRAINED layer (rows = who looks, columns = who is looked at):")
print(w[0].detach())

OK   output keeps the shape (1, 4, 8)
OK   weights are (1, 4, 4): every word looks at every word
OK   3 matrices of 8x8 = 192 parameters
OK   equals the formula written out by hand
attention weights of an UNTRAINED layer (rows = who looks, columns = who is looked at):
tensor([[0.3832, 0.1482, 0.3163, 0.1523],
        [0.2808, 0.2398, 0.2080, 0.2714],
        [0.2603, 0.2310, 0.2535, 0.2552],
        [0.3150, 0.2042, 0.2239, 0.2568]])


## Task 5: Self-attention cannot see order

Shuffle the words of the sentence. Because there is no recurrence, the output rows should simply come out in the same shuffled order, with the SAME values. Test it: apply the layer to `x[:, perm]` and compare with `out[:, perm]`. Then do it again with positions: the words move but the position vectors stay at positions 0 to 3, so the rows should NO LONGER match.

In [ ]:
perm = [2, 0, 3, 1]
out_shuffled_in, _ = sa(x[:, perm])
max_diff = float((out_shuffled_in - out[:, perm]).abs().max().detach())
pe4 = sinusoidal_pe(4, 8)
out_pos, _ = sa(x + pe4)
out_pos_shuf, _ = sa(x[:, perm] + pe4)
max_diff_with_pe = float((out_pos_shuf - out_pos[:, perm]).abs().max().detach())

In [ ]:
print("without positions, max difference:", max_diff)
print("with positions,    max difference:", max_diff_with_pe)
check("without positions the layer is order-blind (difference ~ 0)", max_diff < 1e-5)
check("with positions it is NOT (difference clearly > 0)", max_diff_with_pe > 1e-3)

without positions, max difference: 1.1920928955078125e-07
with positions,    max difference: 0.11804751306772232
OK   without positions the layer is order-blind (difference ~ 0)
OK   with positions it is NOT (difference clearly > 0)


## Task 6: Why divide by sqrt(d_k)?

Make 2000 random query and key vectors with unit-variance entries, for several sizes `d`. Compute the standard deviation of the dot products **without** scaling and **with** the `1/sqrt(d)` scaling. Then take 10 random keys and measure the average of the LARGEST softmax weight in both cases.

In [ ]:
dims = [4, 16, 64, 256, 1024]
std_raw, std_scaled, maxw_raw, maxw_scaled = [], [], [], []
g = torch.Generator().manual_seed(0)
for d in dims:
    q = torch.randn(2000, d, generator=g); k = torch.randn(2000, 10, d, generator=g)
    raw = torch.einsum("nd,nkd->nk", q, k)          # (2000, 10) dot products
    scaled = raw / math.sqrt(d)                     # scaled by dividing by sqrt(d)
    std_raw.append(float(raw.std())); std_scaled.append(float(scaled.std()))
    maxw_raw.append(float(F.softmax(raw, -1).max(-1).values.mean()))
    maxw_scaled.append(float(F.softmax(scaled, -1).max(-1).values.mean()))
for row in zip(dims, std_raw, std_scaled, maxw_raw, maxw_scaled): print("d=%4d  std raw %6.2f  std scaled %5.2f  max weight raw %.2f  scaled %.2f" % row)

d=   4  std raw   2.03  std scaled  1.02  max weight raw 0.49  scaled 0.31
d=  16  std raw   4.01  std scaled  1.00  max weight raw 0.72  scaled 0.32
d=  64  std raw   8.02  std scaled  1.00  max weight raw 0.86  scaled 0.32
d= 256  std raw  16.15  std scaled  1.01  max weight raw 0.93  scaled 0.32
d=1024  std raw  32.15  std scaled  1.00  max weight raw 0.96  scaled 0.32


In [ ]:
check("raw std grows like sqrt(d)", abs(std_raw[-1] / std_raw[0] - math.sqrt(1024 / 4)) < 2.0)
check("scaled std stays near 1 for every d", all(0.8 < s < 1.2 for s in std_scaled))
check("unscaled softmax gets 'one-hot' at large d", maxw_raw[-1] > 0.9)
check("scaled softmax keeps a soft spread", maxw_scaled[-1] < 0.6)

OK   raw std grows like sqrt(d)
OK   scaled std stays near 1 for every d
OK   unscaled softmax gets 'one-hot' at large d
OK   scaled softmax keeps a soft spread


## Task 7: Does position really matter? Measure it
**The task.** Sequences of 8 tokens (vocabulary 12). Token `1` ("A") and token `2` ("B") each appear exactly once at random places; the other 6 tokens are random filler. **Label = 1 if A comes BEFORE B.** Only order decides the answer.

**The model.** Embedding → (optional positions) → ONE self-attention layer with a residual → average over the 8 positions → linear (2 classes). We train three versions: no positions, sinusoidal, learned. 3 seeds each, 4000 training and 1000 validation sequences.

In [ ]:
def make_order_data(n, seed, L=8, V=12):
    g = torch.Generator().manual_seed(seed)
    x = torch.randint(3, V, (n, L), generator=g)
    pa = torch.zeros(n, dtype=torch.long); pb = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        p = torch.randperm(L, generator=g)[:2]
        x[i, p[0]] = 1; x[i, p[1]] = 2; pa[i] = p[0]; pb[i] = p[1]
    return x, (pa < pb).long()
xs, ys = make_order_data(3, 0)
print(xs, ys, "(label 1 means token 1 comes before token 2)")

tensor([[ 1,  3,  5,  9, 10,  2, 10,  4],
        [ 4,  3, 11,  5,  1,  2,  4,  5],
        [ 3,  3,  8,  6, 11,  2, 11,  1]]) tensor([1, 1, 0]) (label 1 means token 1 comes before token 2)


### Task 7a: The classifier

Fill in `forward`. Add positions according to `self.pe_kind` (`'none'`, `'sin'`, `'learned'`), then one self-attention step with a residual, mean pooling, and the output layer.

In [ ]:
class OrderClassifier(nn.Module):
    def __init__(self, pe_kind, d=32, L=8, V=12):
        super().__init__()
        self.pe_kind, self.d = pe_kind, d
        self.emb = nn.Embedding(V, d)
        self.attn = SelfAttention(d)
        self.out = nn.Linear(d, 2)
        if pe_kind == "sin": self.register_buffer("pe", sinusoidal_pe(L, d))
        if pe_kind == "learned": self.pos = nn.Embedding(L, d)
    def forward(self, x):
        h = self.emb(x)
        if self.pe_kind == "sin":
            h = h + self.pe
        elif self.pe_kind == "learned":
            positions = torch.arange(x.size(1), device=x.device)
            h = h + self.pos(positions)
        attn_out, _ = self.attn(h)
        h2 = h + attn_out
        pooled = h2.mean(dim=1)
        return self.out(pooled)

In [ ]:
xb, yb = make_order_data(16, 5)
for kind in ["none", "sin", "learned"]:
    m = OrderClassifier(kind)
    logits = m(xb)
    check(f"{kind}: output shape (16, 2)", tuple(logits.shape) == (16, 2))
m0 = OrderClassifier("none"); idx = torch.randperm(8)
check("'none' version cannot tell a shuffled sequence apart (same logits)", torch.allclose(m0(xb), m0(xb[:, idx]), atol=1e-5))
m1 = OrderClassifier("sin")
check("'sin' version CAN (logits change)", not torch.allclose(m1(xb), m1(xb[:, idx]), atol=1e-3))

OK   none: output shape (16, 2)
OK   sin: output shape (16, 2)
OK   learned: output shape (16, 2)
OK   'none' version cannot tell a shuffled sequence apart (same logits)
OK   'sin' version CAN (logits change)


In [ ]:
def train_order(kind, seed, epochs=30, lr=3e-3):
    torch.manual_seed(seed)
    xt, yt = make_order_data(4000, seed); xv, yv = make_order_data(1000, seed + 100)
    model = OrderClassifier(kind); opt = torch.optim.Adam(model.parameters(), lr)
    for ep in range(epochs):
        perm = torch.randperm(4000)
        for i in range(0, 4000, 64):
            b = perm[i:i + 64]
            loss = F.cross_entropy(model(xt[b]), yt[b]); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): return float((model(xv).argmax(1) == yv).float().mean())
t0 = time.time(); order = {}
for kind in ["none", "sin", "learned"]:
    order[kind] = [train_order(kind, s) for s in (0, 1, 2)]
    print(f"{kind:8s} validation accuracy per seed:", [round(a, 3) for a in order[kind]])
print("took %.0f s" % (time.time() - t0)); R["order"] = order

none     validation accuracy per seed: [0.503, 0.479, 0.53]
sin      validation accuracy per seed: [1.0, 1.0, 1.0]
learned  validation accuracy per seed: [1.0, 1.0, 1.0]
took 63 s


### Task 7b: Read the result

Set each variable to `True` or `False` from the numbers above (chance level is 0.5).

In [ ]:
none_is_chance = True       # The 'none' model's mean accuracy is within 0.05 of 0.5
sin_solves_it = True        # The mean accuracy of 'sin' is above 0.9
learned_solves_it = True    # The mean accuracy of 'learned' is above 0.9

In [ ]:
mean = lambda a: sum(a) / len(a)
check("none", none_is_chance == (abs(mean(order["none"]) - 0.5) < 0.05))
check("sin", sin_solves_it == (mean(order["sin"]) > 0.9)); check("learned", learned_solves_it == (mean(order["learned"]) > 0.9))

OK   none
OK   sin
OK   learned


## Wrap-up
- Self-attention: every word looks at every word with `softmax(QKᵀ/√d_k) V`.
- It has no sense of order; positions must be added to the embeddings.
- `√d_k` keeps the softmax from collapsing to a one-hot as `d_k` grows.
- Next lecture: several heads in parallel, residual connections, layer norm, the feed-forward layer, and a full encoder block.

**Think about it:** which of Page 4's two position methods would you pick if test sentences are much longer than training ones, and why? (Our task above cannot answer that. We only used length 8.)

In [ ]:
print("RESULT_JSON " + json.dumps(R))

RESULT_JSON {"order": {"none": [0.503000020980835, 0.4790000021457672, 0.5299999713897705], "sin": [1.0, 1.0, 1.0], "learned": [1.0, 1.0, 1.0]}}
